# OCR Experiments

Explore the project’s EasyOCR flow on image files and scanned PDF pages, and compare OCR on an image before and after the project’s preprocessing. Run `uv sync` from the project root and select the project’s Python environment as this notebook’s kernel. EasyOCR runs on CPU and may download its model the first time it is initialized.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project root (pyproject.toml) from the current directory.")

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from IPython.display import display
from HandwrittenNotesRAG.components.image_loader import ImageLoader
from HandwrittenNotesRAG.components.image_preprocessing import ImagePreprocessor
from HandwrittenNotesRAG.components.ocr_extractor import OCRExtractor
from HandwrittenNotesRAG.components.pdf_loader import PDFLoader

pdf_dir = project_root / "data" / "raw" / "pdfs"
image_dir = project_root / "data" / "raw" / "images"
pdf_files = sorted(pdf_dir.glob("*.pdf"))
image_files = sorted(
    path for path in image_dir.iterdir()
    if path.is_file() and path.suffix.lower() in ImageLoader.SUPPORTED_EXTENSIONS
) if image_dir.is_dir() else []

print(f"Project root: {project_root}")
print(f"PDF files: {len(pdf_files)}; image files: {len(image_files)}")

## Compare image preprocessing

By default, the first supported image in `data/raw/images/` is selected. Change `image_path` to experiment with another file. The project preprocesses images to grayscale, applies autocontrast, and limits their size before OCR.

In [ ]:
image_path = image_files[0] if image_files else None
image = ImageLoader().load(image_path) if image_path is not None else None

if image is None:
    print("No supported image found. Add one to data/raw/images/ and rerun this cell.")
else:
    print(f"Selected image: {image_path.name}")
    display(image)

In [ ]:
ocr = OCRExtractor(languages=("en",)) if image is not None or pdf_files else None
preprocessor = ImagePreprocessor()

if image is None:
    print("Load an image in the previous cell to compare its OCR output before and after preprocessing.")
else:
    processed_image = preprocessor.preprocess(image)
    display(processed_image)

    raw_text = ocr.extract_text(image)
    processed_text = ocr.extract_text(processed_image)
    print("OCR on original image:\n")
    print(raw_text or "No text detected.")
    print("\nOCR after preprocessing:\n")
    print(processed_text or "No text detected.")

## Read a PDF page

`PDFLoader` returns embedded page text directly and renders image-only pages for OCR. Change `pdf_path` or `page_number` to inspect another page. Page numbers are one-based.

In [ ]:
pdf_path = pdf_files[0] if pdf_files else None
page_number = 1

if pdf_path is None:
    print("No PDF found. Add one to data/raw/pdfs/ and rerun this cell.")
else:
    pages = PDFLoader().load_pdf(pdf_path)
    if not 1 <= page_number <= len(pages):
        raise ValueError(f"page_number must be between 1 and {len(pages)}")

    page = pages[page_number - 1]
    print(f"{pdf_path.name}, page {page_number}")
    if page["text"]:
        print("Embedded PDF text:\n")
        print(page["text"])
    elif page["image"] is not None and ocr is not None:
        display(page["image"])
        page_image = preprocessor.preprocess(page["image"])
        display(page_image)
        print("OCR text:\n")
        print(ocr.extract_text(page_image) or "No text detected.")
    else:
        print("No text or page image was available.")

OCR output is filtered by the extractor’s confidence threshold. Compare the results with the source image or a known transcription; this notebook displays examples but does not calculate accuracy without ground-truth text.